# Mukasa: 10 DQN experiments on Pong

This notebook is only for **Mukasa**. It trains the 10 settings in `MukasaHyperparameter_experiment_results.md`. It does not train Emmanuel-NS's settings or the other member's settings.

The game is **ALE/Pong-v5**. Each run is **120 episodes**. The code is the group's own `train.py`. Nothing here uses Stable-Baselines3, RLlib, CleanRL, or TorchRL.

## Before you run

1. In Colab, choose **Runtime → Change runtime type → T4 GPU → Save**.
2. Upload this notebook file itself: `colab_train_mukasa.ipynb`. Do not reopen an older copy of the notebook. The older copy does not save to Google Drive.
3. Keep the tab open. A Colab disconnect deletes `/content`, including every model that was not copied to Drive.
4. Each finished experiment is copied to `My Drive/formative-3-dqn/Mukasa/`. If the runtime stops, run the cells again. An experiment that is already on Drive is restored and skipped.
5. When experiment 10 prints `saved to Drive`, download that Drive folder into the repository as `models/Mukasa/`.


## Your 10 settings

Experiment 1 is the baseline. Every other row changes one value.


| Experiment | What changes from experiment 1 | Learning rate | Gamma | Batch size | Epsilon start | Epsilon end | Epsilon decay |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | Baseline | 0.00025 | 0.95 | 64 | 1.0 | 0.1 | 0.995 |
| 2 | Higher learning rate | 0.0005 | 0.95 | 64 | 1.0 | 0.1 | 0.995 |
| 3 | Lower learning rate | 0.00005 | 0.95 | 64 | 1.0 | 0.1 | 0.995 |
| 4 | Shorter reward horizon | 0.00025 | 0.9 | 64 | 1.0 | 0.1 | 0.995 |
| 5 | Longer reward horizon | 0.00025 | 0.99 | 64 | 1.0 | 0.1 | 0.995 |
| 6 | Smaller batch | 0.00025 | 0.95 | 16 | 1.0 | 0.1 | 0.995 |
| 7 | Larger batch | 0.00025 | 0.95 | 128 | 1.0 | 0.1 | 0.995 |
| 8 | Less exploration at the start | 0.00025 | 0.95 | 64 | 0.3 | 0.1 | 0.995 |
| 9 | Lower final exploration | 0.00025 | 0.95 | 64 | 1.0 | 0.01 | 0.995 |
| 10 | Faster epsilon decay | 0.00025 | 0.95 | 64 | 1.0 | 0.1 | 0.98 |


## 1. Mount Google Drive

Allow the sign-in window. This has to succeed before training starts.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
print("Drive is mounted. Models will be saved under My Drive/formative-3-dqn/")


## 2. Name and settings

Leave `MEMBER` as it is. These values are this notebook's plan.


In [ ]:
import os

os.environ["MPLBACKEND"] = "Agg"
MEMBER = "Mukasa"
EPISODES = 120

PLAN = [
    {"n": 1, "lr": "0.00025", "gamma": "0.95", "batch": 64, "eps_start": "1.0", "eps_end": "0.1", "eps_decay": "0.995"},
    {"n": 2, "lr": "0.0005", "gamma": "0.95", "batch": 64, "eps_start": "1.0", "eps_end": "0.1", "eps_decay": "0.995"},
    {"n": 3, "lr": "0.00005", "gamma": "0.95", "batch": 64, "eps_start": "1.0", "eps_end": "0.1", "eps_decay": "0.995"},
    {"n": 4, "lr": "0.00025", "gamma": "0.9", "batch": 64, "eps_start": "1.0", "eps_end": "0.1", "eps_decay": "0.995"},
    {"n": 5, "lr": "0.00025", "gamma": "0.99", "batch": 64, "eps_start": "1.0", "eps_end": "0.1", "eps_decay": "0.995"},
    {"n": 6, "lr": "0.00025", "gamma": "0.95", "batch": 16, "eps_start": "1.0", "eps_end": "0.1", "eps_decay": "0.995"},
    {"n": 7, "lr": "0.00025", "gamma": "0.95", "batch": 128, "eps_start": "1.0", "eps_end": "0.1", "eps_decay": "0.995"},
    {"n": 8, "lr": "0.00025", "gamma": "0.95", "batch": 64, "eps_start": "0.3", "eps_end": "0.1", "eps_decay": "0.995"},
    {"n": 9, "lr": "0.00025", "gamma": "0.95", "batch": 64, "eps_start": "1.0", "eps_end": "0.01", "eps_decay": "0.995"},
    {"n": 10, "lr": "0.00025", "gamma": "0.95", "batch": 64, "eps_start": "1.0", "eps_end": "0.1", "eps_decay": "0.98"},
]

print(MEMBER, "will train these 10 settings:")
for row in PLAN:
    changed = []
    baseline = PLAN[0]
    for key in ("lr", "gamma", "batch", "eps_start", "eps_end", "eps_decay"):
        if row[key] != baseline[key]:
            changed.append(key + "=" + str(row[key]))
    label = "baseline" if row["n"] == 1 else ", ".join(changed)
    print(row["n"], label)


## 3. Install

When the file chooser opens, upload `formative-3-dqn-colab.zip` from the project folder. The cell installs uv, the project packages, and the Atari ROM.


In [ ]:
import os
import zipfile
import subprocess
from pathlib import Path
from google.colab import files

os.environ["MPLBACKEND"] = "Agg"
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + os.pathsep + os.environ["PATH"]

if not Path("pyproject.toml").exists():
    print("Upload formative-3-dqn-colab.zip when the file chooser opens.")
    uploaded = files.upload()
    for name in uploaded:
        if name.endswith(".zip"):
            zipfile.ZipFile(name).extractall(".")
    found = next(Path(".").rglob("pyproject.toml"), None)
    if found is None:
        raise SystemExit("pyproject.toml was not in the upload.")
    os.chdir(found.parent)

print("Project folder:", Path(".").resolve())
subprocess.run("curl -LsSf https://astral.sh/uv/install.sh | sh", shell=True, check=True)
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + os.pathsep + os.environ["PATH"]
subprocess.run("apt-get install -y -qq libgl1 libglib2.0-0", shell=True, check=True)
subprocess.run("uv sync", shell=True, check=True)
subprocess.run("uv run AutoROM --accept-license", shell=True, check=True)
print("Install finished.")


## 4. Train

The first line of the log must say `cuda True` and `Device: cuda`. After each experiment, wait for `saved to Drive` before leaving the tab. That line means the model is on Google Drive, not only on the Colab disk.


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

os.environ["MPLBACKEND"] = "Agg"
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + os.pathsep + os.environ["PATH"]
os.environ["PYTHONUNBUFFERED"] = "1"

project = next(Path("/content").rglob("train.py")).parent
os.chdir(project)
drive_root = Path("/content/drive/MyDrive/formative-3-dqn") / MEMBER
local_root = project / "models" / MEMBER
print("Using", project)
print("Drive folder", drive_root)

if not Path("/content/drive/MyDrive").exists():
    raise SystemExit("Google Drive is not mounted. Run the Drive cell first.")

# A previous runtime may already have saved some experiments to Drive.
if drive_root.exists():
    shutil.copytree(drive_root, local_root, dirs_exist_ok=True)
    print("Restored finished experiments from Drive.")

probe = subprocess.run(
    [
        "uv", "run", "python", "-u", "-c",
        "import torch; print('cuda', torch.cuda.is_available()); "
        "from dqn import make_env; env = make_env(); "
        "print('actions', env.action_space.n); env.close()",
    ],
    cwd=project,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(probe.stdout)
if probe.returncode != 0:
    raise SystemExit(probe.returncode)
if "cuda True" not in probe.stdout:
    raise SystemExit(
        "This runtime has no GPU. Choose Runtime, Change runtime type, T4 GPU, then run this cell again."
    )

for row in PLAN:
    folder = local_root / f"experiment_{row['n']:02d}"
    summary = folder / "summary.json"
    if summary.exists():
        print("skip", row["n"], "already saved", flush=True)
        continue
    command = [
        "uv", "run", "python", "-u", "train.py",
        "--member", MEMBER,
        "--experiment", str(row["n"]),
        "--episodes", str(EPISODES),
        "--learning-rate", row["lr"],
        "--gamma", row["gamma"],
        "--batch-size", str(row["batch"]),
        "--epsilon-start", row["eps_start"],
        "--epsilon-end", row["eps_end"],
        "--epsilon-decay", row["eps_decay"],
    ]
    print("start", row["n"], flush=True)
    process = subprocess.Popen(
        command,
        cwd=project,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for line in process.stdout:
        print(line, end="")
    if process.wait() != 0:
        raise SystemExit(f"experiment {row['n']} failed")
    destination = drive_root / folder.name
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(folder, destination, dirs_exist_ok=True)
    print("saved to Drive", row["n"], destination, flush=True)

print("finished", MEMBER)
print("Download My Drive/formative-3-dqn/" + MEMBER + " into the repository as models/" + MEMBER)


## After the last `saved to Drive`

1. Open Google Drive on this computer.
2. Download `My Drive/formative-3-dqn/Mukasa`.
3. Put it in the repository so this file exists: `models/Mukasa/experiment_01/dqn_model.pth`, and the same for experiments 02 through 10.
4. Each folder also needs `training_performance.png`, `training_log.csv`, `summary.json`, and `config.json`.
5. Copy each printed line `Average reward over the last 20 episodes` into `MukasaHyperparameter_experiment_results.md`.

Do not copy another member's model files into this folder.
